# IAD Pipeline
Anomaly detection pipeline using FiftyOne, Weights & Biases, and the IAD framework.

## 1. Environment Setup
Configure database URI and API keys.

In [1]:
import os
import sys
import warnings
import yaml
# Set BEFORE any fiftyone imports
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost"
os.environ["WANDB_API_KEY"] = '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'
sys.path.append("..")

# Now safe to import
import wandb
import logging
from pathlib import Path
from src.manager import AnomalyDetectionManager as ADM
from src.manager import DatasetSession as DS

wandb.login()

W0622 15:58:53.850000 19204 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\Admin\Documents\AnomalyDetection\.venv\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
wandb: Currently logged in as: daniel-pommer (daniel-pommer-technische-hochschule-n-rnberg-georg-simon-ohm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2. Configuration
Set your run parameters here before executing the pipeline.

In [ ]:
logger              = logging.getLogger("logger")

productName         = "bottle"
datasetName         = "MVTecADShortPred"
split               = ("pred",)

datasetDir          = Path("../datasets/")
configDir           = Path("../configs/")
outputPath          = Path("../results/")
productConfigPath   = Path(f"Products/{productName}.yaml")
productConfigPath=Path(configDir/productConfigPath)

manager, productDescription = ADM.loadProduct(productConfigPath=productConfigPath, outputPath=outputPath, configDir=configDir)
datasetSession = DS.loadDatasetFromDisk(datasetDir/datasetName, datasetName, split=split)
datasetSession.select_category(productName)
manager.adjustPaths(datasetName=datasetName, category=productName, adjustCheckpoints=False) # We want to use the checkpoints of the training dataset not the new predictions dataset (does not have checkpoints)
print(f"Output path: {manager.outputPath}")
print(f"Checkpoint path: {manager.ckptDir}")

## 3. Inspect Dataset

In [ ]:
datasetSession.launchSession()

## 4. Prediction

In [ ]:
if manager.ckptPath is not None:
    if not manager.isTilingSetup:
        manager.loadCheckpoint(manager.ckptPath, f"{manager.modelName}")
    if manager.isTilingSetup:
        manager.setupTiling(configDir / "Tiling" / "TiledEnsemblePred.yaml")
    if manager.inferencerPath is not None:
        manager.inference(datasetSession=datasetSession,
                          inferenceConfigPath=manager.inferencerPath,
                          resultsDir=manager.outputPath,
                          tiling=manager.isTilingSetup,
                          ckptPath=manager.ckptDir,
                          trainingDir=productDescription["model"]["trainingDir"])
    print(manager.FO_Dataset)


In [ ]:
manager.launchSession()
